In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# Sex by age bands
# To download original dataset go to - 
# https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221
student_acc_by_age_csv = "N:\Geodatabase\Raw_Data\Census 2021\Students\RM129 - Student accommodation by age.csv"

## 2. Input Link to LSOA 2021 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::lower-layer-super-output-areas-december-2021-boundaries-ew-bsc-v4-2/about

In [3]:
lsoa2021_shapefile = r"C:\Users\abhimanya.achara\OneDrive - priorpartners.com\Documents\UK DATA\2021 Census Data\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW\Lower_Layer_Super_Output_Areas_(Dec_2021)_Boundaries_Full_Extent_(BFE)_EW.shp"

## 3. Threshold to define dominant type
This defines the % threshold from the highest value of group of data columns to define which ones will be defined as dominant. 

In [4]:
threshold_value = 2.5

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [5]:
def add_percentage_columns(df, total_col, suffix='_count', new_suffix='_perc', columns_to_convert=None):
    
    if columns_to_convert is None:
        columns_to_convert = [col for col in df.columns if col.endswith(suffix) and col != total_col]

    for col in columns_to_convert:
        perc_col = col.replace(suffix, new_suffix)
        df[perc_col] = (df[col] / df[total_col]) * 100

    return df

In [6]:
# Create Dataframes
student_by_age_df = pd.read_csv(student_acc_by_age_csv, skiprows = 8, nrows = 35672, skip_blank_lines = True)
student_by_age_living_with_parents_df = pd.read_csv(student_acc_by_age_csv, skiprows = 35704, nrows = 35672, skip_blank_lines = True)
student_by_age_living_communal_uni_df = pd.read_csv(student_acc_by_age_csv, skiprows = 107096, nrows = 35672, skip_blank_lines = True)
student_by_age_living_communal_other_df = pd.read_csv(student_acc_by_age_csv, skiprows = 142792, nrows = 35672, skip_blank_lines = True)
student_by_age_living_all_student_df = pd.read_csv(student_acc_by_age_csv, skiprows = 178488, nrows = 35672, skip_blank_lines = True)
student_by_age_living_alone_df = pd.read_csv(student_acc_by_age_csv, skiprows = 214184, nrows = 35672, skip_blank_lines = True)
student_by_age_living_other_household_df = pd.read_csv(student_acc_by_age_csv, skiprows = 249880, nrows = 35672, skip_blank_lines = True)

df_list = [
    student_by_age_df,
    student_by_age_living_with_parents_df,
    student_by_age_living_communal_uni_df,
    student_by_age_living_communal_other_df,
    student_by_age_living_all_student_df,
    student_by_age_living_alone_df,
    student_by_age_living_other_household_df
]

student_by_age_living_with_parents_df.head()

,2021 super output area - lower layer,Total,Aged 4 years and under,Aged 5 to 15 years,Aged 16 to 17 years,Aged 18 to 20 years,Aged 21 to 24 years,Aged 25 to 29 years,Aged 30 years and over
0,E01011954 : Hartlepool 001A,460,0,358,53,28,17,3,1
1,E01011969 : Hartlepool 001B,168,0,132,19,12,4,1,0
2,E01011970 : Hartlepool 001C,117,0,88,16,8,5,0,0
3,E01011971 : Hartlepool 001D,268,0,191,39,18,15,4,1
4,E01033465 : Hartlepool 001F,469,0,357,64,38,9,1,0


In [7]:
# Dictionary for renaming columns with corrected keys
column_rename_map1 = {
    "total_count": "total_students",
    "aged_4_years_and_under_count": "age_4_under_count",
    "aged_5_to_15_years_count": "age_5_15_count",
    "aged_16_to_17_years_count": "age_16_17_count",    
    "aged_18_to_20_years_count": "age_18_20_count",
    "aged_21_to_24_years_count": "age_21_24_count",
    "aged_25_to_29_years_count": "age_25_29_count",
    "aged_30_years_and_over_count": "age_30_over_count"
}

column_rename_map2 = {
    "total_count": "living_with_parents_count",
    "aged_4_years_and_under_count": "living_with_parents_age_4_under_count",
    "aged_5_to_15_years_count": "living_with_parents_age_5_15_count",
    "aged_16_to_17_years_count": "living_with_parents_age_16_17_count",    
    "aged_18_to_20_years_count": "living_with_parents_age_18_20_count",
    "aged_21_to_24_years_count": "living_with_parents_age_21_24_count",
    "aged_25_to_29_years_count": "living_with_parents_age_25_29_count",
    "aged_30_years_and_over_count": "living_with_parents_age_30_over_count"
}

column_rename_map3 = {
    "total_count": "living_in_communal_establishment_uni_count",
    "aged_4_years_and_under_count": "living_in_communal_establishment_uni_age_4_under_count",
    "aged_5_to_15_years_count": "living_in_communal_establishment_uni_age_5_15_count",
    "aged_16_to_17_years_count": "living_in_communal_establishment_uni_age_16_17_count",    
    "aged_18_to_20_years_count": "living_in_communal_establishment_uni_age_18_20_count",
    "aged_21_to_24_years_count": "living_in_communal_establishment_uni_age_21_24_count",
    "aged_25_to_29_years_count": "living_in_communal_establishment_uni_age_25_29_count",
    "aged_30_years_and_over_count": "living_in_communal_establishment_uni_age_30_over_count"
}

column_rename_map4 = {
    "total_count": "living_in_communal_establishment_other_count",
    "aged_4_years_and_under_count": "living_in_communal_establishment_other_age_4_under_count",
    "aged_5_to_15_years_count": "living_in_communal_establishment_other_age_5_15_count",
    "aged_16_to_17_years_count": "living_in_communal_establishment_other_age_16_17_count",    
    "aged_18_to_20_years_count": "living_in_communal_establishment_other_age_18_20_count",
    "aged_21_to_24_years_count": "living_in_communal_establishment_other_age_21_24_count",
    "aged_25_to_29_years_count": "living_in_communal_establishment_other_age_25_29_count",
    "aged_30_years_and_over_count": "living_in_communal_establishment_other_age_30_over_count"
}

column_rename_map5 = {
    "total_count": "living_in_all_student_household_count",
    "aged_4_years_and_under_count": "living_in_all_student_household_age_4_under_count",
    "aged_5_to_15_years_count": "living_in_all_student_household_age_5_15_count",
    "aged_16_to_17_years_count": "living_in_all_student_household_age_16_17_count",    
    "aged_18_to_20_years_count": "living_in_all_student_household_age_18_20_count",
    "aged_21_to_24_years_count": "living_in_all_student_household_age_21_24_count",
    "aged_25_to_29_years_count": "living_in_all_student_household_age_25_29_count",
    "aged_30_years_and_over_count": "living_in_all_student_household_age_30_over_count"
}

column_rename_map6 = {
    "total_count": "living_alone_count",
    "aged_4_years_and_under_count": "living_alone_age_4_under_count",
    "aged_5_to_15_years_count": "living_alone_age_5_15_count",
    "aged_16_to_17_years_count": "living_alone_age_16_17_count",    
    "aged_18_to_20_years_count": "living_alone_age_18_20_count",
    "aged_21_to_24_years_count": "living_alone_age_21_24_count",
    "aged_25_to_29_years_count": "living_alone_age_25_29_count",
    "aged_30_years_and_over_count": "living_alone_age_30_over_count"
}

column_rename_map7 = {
    "total_count": "living_in_other_household_count",
    "aged_4_years_and_under_count": "living_in_other_household_age_4_under_count",
    "aged_5_to_15_years_count": "living_in_other_household_age_5_15_count",
    "aged_16_to_17_years_count": "living_in_other_household_age_16_17_count",    
    "aged_18_to_20_years_count": "living_in_other_household_age_18_20_count",
    "aged_21_to_24_years_count": "living_in_other_household_age_21_24_count",
    "aged_25_to_29_years_count": "living_in_other_household_age_25_29_count",
    "aged_30_years_and_over_count": "living_in_other_household_age_30_over_count"
}

rename_list = [column_rename_map1,
               column_rename_map2,
               column_rename_map3,
               column_rename_map4,
               column_rename_map5,
               column_rename_map6,
               column_rename_map7
]

totals_list = ["total_students",
               "living_with_parents_count",
               "living_in_communal_establishment_uni_count",
               "living_in_communal_establishment_other_count",
               "living_in_all_student_household_count",
               "living_alone_count",
               "living_in_other_household_count" 
]             
               
               
            

In [8]:
for i, df in enumerate(df_list):
    # Split the lsoa column into two new columns
    df[['lsoa21cd', 'lsoa21nm']] = df.iloc[:, 0].str.split(' : ', expand=True)
    
    # Remove the column '2021 super output area - lower layer'
    df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
    
    # Rename columns 
    cols_to_rename = df.columns.difference(['lsoa21cd', 'lsoa21nm'])
    df.rename(columns={col: col.lower().replace(' ', '_') + '_count' for col in cols_to_rename}, inplace=True)
    
    # Remove the column '2021 super output area - lower layer'
    df.drop(['lsoa21nm'], 1,  inplace=True)
    
    # Move 'lsoa21cd' and 'lsoa21nm' to the front of the dataframe
    cols = df.columns.tolist()
    new_order = [cols[-1]] + cols[:-1]
    df_list[i] = df[new_order]   
    
    # Rename columns using the dictionary
    df_list[i].rename(columns=rename_list[i], inplace=True)

    # Add percentage column
    add_percentage_columns(df_list[i], totals_list[i], suffix='_count', new_suffix='_perc', columns_to_convert=df_list[i].columns[-7:])



C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\3978777113.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\3978777113.py:13: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['lsoa21nm'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\3978777113.py:6: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  df.drop(['2021 super output area - lower layer'], 1,  inplace=True)
C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\3978777113.py:13: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels

In [9]:
from functools import reduce

# Merge all DataFrames in df_list on 'lsoa21cd', keeping the order from the first df
merged_df = reduce(
    lambda left, right: pd.merge(left, right, on='lsoa21cd', how='left'),
    df_list
)

merged_df.head()

,lsoa21cd,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_count,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_count,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_count,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_count,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_count,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household_age_16_17_count,living_in_other_household_age_18_20_count,living_in_other_household_age_21_24_count,living_in_other_household_age_25_29_count,living_in_other_household_age_30_over_count,living_in_other_household_age_4_under_perc,living_in_other_household_age_5_15_perc,living_in_other_household_age_16_17_perc,living_in_other_household_age_18_20_per

In [10]:
for col in totals_list:
    if col != "total_students":
        perc_col = col.replace('_count', '_perc')
        merged_df[perc_col] = (merged_df[col] / merged_df["total_students"]) * 100

merged_df.head()

,lsoa21cd,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_count,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_count,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_count,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_count,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_count,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household_age_16_17_count,living_in_other_household_age_18_20_count,living_in_other_household_age_21_24_count,living_in_other_household_age_25_29_count,living_in_other_household_age_30_over_count,living_in_other_household_age_4_under_perc,living_in_other_household_age_5_15_perc,living_in_other_household_age_16_17_perc,living_in_other_household_age_18_20_per

## 4. Load GIS shapefile 

In [11]:
# Attempt to read from the server first, fallback to local path if it fails
lsoa2021boundary_df = gpd.read_file(lsoa2021_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
lsoa2021boundary_df.head()

Shapefile loaded successfully from the server.


,FID,LSOA21CD,LSOA21NM,Shape__Are,Shape__Len,GlobalID,geometry
0,1,E01000001,City of London 001A,129865.314476,2635.767993,4dd060c0-a44a-4c62-aca3-b0c88c0bf0d0,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,228419.782242,2707.816821,d0a47b5d-5649-4242-a8e7-462078975c1d,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,59054.204697,1224.573160,2904cc10-e994-4a6e-b11c-06be7fbd74d2,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,189577.709503,2275.805344,bfcf5958-f052-4388-8efd-75bb808d8b83,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,146536.995750,1966.092607,64a24862-79c1-4c8c-ab7b-d9cec1d3c0c6,"POLYGON ((545126.852 184310.838, 545145.213 18..."


## 5. GIS data management

### Remove Rename fields

In [12]:
#Drop and rename fields
lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)
lsoa2021boundary_df.rename(columns = {'LSOA21CD':'lsoa21cd','LSOA21NM':'lsoa21nm'}, inplace = True)
lsoa2021boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\2026090638.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoa2021boundary_df.drop(['Shape__Are', 'Shape__Len', 'GlobalID'], 1, inplace = True)


,FID,lsoa21cd,lsoa21nm,geometry
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18..."
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18..."
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18..."
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18..."
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [13]:
# Define the file path for msoa11 to msoa21 lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\OA21_LSOA21_MSOA21_LAD22_EW_LU.csv"

# Read the Excel file
lsoalookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop oa column
lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)

#remove duplicates
lsoalookup_df = lsoalookup_df.drop_duplicates(subset='lsoa21cd', keep='first')

# Display the first few rows
lsoalookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\1268765203.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoalookup_df.drop(['oa21cd','lsoa21nm'],1,inplace = True)


,lsoa21cd,msoa21cd,msoa21nm,lad22cd,lad22nm
0,E01000001,E02000001,City of London 001,E09000001,City of London
4,E01000003,E02000001,City of London 001,E09000001,City of London
6,E01000002,E02000001,City of London 001,E09000001,City of London
12,E01032739,E02000001,City of London 001,E09000001,City of London
13,E01032740,E02000001,City of London 001,E09000001,City of London


In [14]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoalookup_df, how = 'left', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham


#### LSOA21 to WARD22

In [15]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.csv"

# Read the Excel file
lsoawardlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)

# Rename fields
lsoawardlookup_df.rename(
    columns={
        'LSOA21CD': 'lsoa21cd',
        'WD22CD': 'wd22cd',
        'WD22NM': 'wd22nm',
        }, 
    inplace=True
)

# Display the first few rows
lsoawardlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\1707502962.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  lsoawardlookup_df.drop(['LSOA21NM','LSOA21NMW','WD22NMW','LAD22NMW','ObjectId','LAD22CD','LAD22NM'],1,inplace = True)


,lsoa21cd,wd22cd,wd22nm
0,E01004766,E05000650,Astley Bridge
1,E01005347,E05000722,Chadderton South
2,E01004890,E05000661,Horwich North East
3,E01004770,E05000650,Astley Bridge
4,E01004888,E05000661,Horwich North East


In [16]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, lsoawardlookup_df, how = 'inner', on = 'lsoa21cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury


#### LAD22 to REGION22

In [17]:
# Define the file path for msoa21 to ward lookup
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_14364\4188529333.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [18]:
lsoa2021boundary_df = pd.merge(lsoa2021boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London


### Add data management fields

In [19]:
# Add new data management fields with specified values
lsoa2021boundary_df['data_source'] = "Census 2021 (ONS Nomis - Official Census and Labour Market Statistics)"
lsoa2021boundary_df['data_resolution'] = "LSOA 2021"
lsoa2021boundary_df['data_time_period'] = pd.to_datetime("2021-02-21")  # Convert to date format
lsoa2021boundary_df['data_web_link'] = "https://www.nomisweb.co.uk/"
lsoa2021boundary_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/


### Update Area

In [20]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in lsoa2021boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    lsoa2021boundary_df['area_ha'] = lsoa2021boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in lsoa2021boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
lsoa2021boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,1,E01000001,City of London 001A,"POLYGON ((532151.537 181867.433, 532152.500 18...",E02000001,City of London 001,E09000001,City of London,E05009288,Aldersgate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,12.986531
1,2,E01000002,City of London 001B,"POLYGON ((532634.497 181926.016, 532632.048 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,22.841978
2,3,E01000003,City of London 001C,"POLYGON ((532153.703 182165.155, 532158.250 18...",E02000001,City of London 001,E09000001,City of London,E05009302,Cripplegate,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,5.905420
3,4,E01000005,City of London 001E,"POLYGON ((533619.062 181402.364, 533639.868 18...",E02000001,City of London 001,E09000001,City of London,E05009308,Portsoken,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,18.957771
4,5,E01000006,Barking and Dagenham 016A,"POLYGON ((545126.852 184310.838, 545145.213 18...",E02000017,Barking and Dagenham 016,E09000002,Barking and Dagenham,E05014066,Northbury,E12000007,London,Census 2021 (ONS Nomis - Official Census and L...,LSOA 2021,2021-02-21,https://www.nomisweb.co.uk/,14.653700


# 7. Combine Geometry and data

In [21]:
census2021_student_lsoa2021_gdb_df = pd.merge(lsoa2021boundary_df,merged_df, how = 'left', on='lsoa21cd')
census2021_student_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_count,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_count,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_count,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_count,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_count,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household_age_16_17_count,living_in_other_household_age_18_20_count,living_in_other_household_age_21_24_count,living_in_other_household_age_25_29_count,living_in_other_household_age_30_over_count,living_in_

# 8. Final tweaks

In [23]:
# Reorder columns in the combined dataframe

final_column_order = ['FID',
 'lsoa21cd',
 'lsoa21nm',
 'geometry',
 'msoa21cd',
 'msoa21nm',
 'lad22cd',
 'lad22nm',
 'wd22cd',
 'wd22nm',
 'rgn22cd',
 'rgn22nm',
 'data_source',
 'data_resolution',
 'data_time_period',
 'data_web_link',
 'area_ha',
 'total_students', 
 'age_4_under_count',
 'age_5_15_count',
 'age_16_17_count',
 'age_18_20_count',
 'age_21_24_count',
 'age_25_29_count',
 'age_30_over_count',
 'age_4_under_perc',
 'age_5_15_perc',
 'age_16_17_perc',
 'age_18_20_perc',
 'age_21_24_perc',
 'age_25_29_perc',
 'age_30_over_perc',
 'living_with_parents_count',
 'living_in_communal_establishment_uni_count',
 'living_in_communal_establishment_other_count',
 'living_in_all_student_household_count',
 'living_alone_count',
 'living_in_other_household_count',
 'living_with_parents_perc',
 'living_in_communal_establishment_uni_perc',
 'living_in_communal_establishment_other_perc',
 'living_in_all_student_household_perc',
 'living_alone_perc',
 'living_in_other_household_perc', 
 'living_with_parents_age_4_under_count',
 'living_with_parents_age_5_15_count',
 'living_with_parents_age_16_17_count',
 'living_with_parents_age_18_20_count',
 'living_with_parents_age_21_24_count',
 'living_with_parents_age_25_29_count',
 'living_with_parents_age_30_over_count',
 'living_with_parents_age_4_under_perc',
 'living_with_parents_age_5_15_perc',
 'living_with_parents_age_16_17_perc',
 'living_with_parents_age_18_20_perc',
 'living_with_parents_age_21_24_perc',
 'living_with_parents_age_25_29_perc',
 'living_with_parents_age_30_over_perc', 
 'living_in_communal_establishment_uni_age_4_under_count',
 'living_in_communal_establishment_uni_age_5_15_count',
 'living_in_communal_establishment_uni_age_16_17_count',
 'living_in_communal_establishment_uni_age_18_20_count',
 'living_in_communal_establishment_uni_age_21_24_count',
 'living_in_communal_establishment_uni_age_25_29_count',
 'living_in_communal_establishment_uni_age_30_over_count',
 'living_in_communal_establishment_uni_age_4_under_perc',
 'living_in_communal_establishment_uni_age_5_15_perc',
 'living_in_communal_establishment_uni_age_16_17_perc',
 'living_in_communal_establishment_uni_age_18_20_perc',
 'living_in_communal_establishment_uni_age_21_24_perc',
 'living_in_communal_establishment_uni_age_25_29_perc',
 'living_in_communal_establishment_uni_age_30_over_perc', 
 'living_in_communal_establishment_other_age_4_under_count',
 'living_in_communal_establishment_other_age_5_15_count',
 'living_in_communal_establishment_other_age_16_17_count',
 'living_in_communal_establishment_other_age_18_20_count',
 'living_in_communal_establishment_other_age_21_24_count',
 'living_in_communal_establishment_other_age_25_29_count',
 'living_in_communal_establishment_other_age_30_over_count',
 'living_in_communal_establishment_other_age_4_under_perc',
 'living_in_communal_establishment_other_age_5_15_perc',
 'living_in_communal_establishment_other_age_16_17_perc',
 'living_in_communal_establishment_other_age_18_20_perc',
 'living_in_communal_establishment_other_age_21_24_perc',
 'living_in_communal_establishment_other_age_25_29_perc',
 'living_in_communal_establishment_other_age_30_over_perc', 
 'living_in_all_student_household_age_4_under_count',
 'living_in_all_student_household_age_5_15_count',
 'living_in_all_student_household_age_16_17_count',
 'living_in_all_student_household_age_18_20_count',
 'living_in_all_student_household_age_21_24_count',
 'living_in_all_student_household_age_25_29_count',
 'living_in_all_student_household_age_30_over_count',
 'living_in_all_student_household_age_4_under_perc',
 'living_in_all_student_household_age_5_15_perc',
 'living_in_all_student_household_age_16_17_perc',
 'living_in_all_student_household_age_18_20_perc',
 'living_in_all_student_household_age_21_24_perc',
 'living_in_all_student_household_age_25_29_perc',
 'living_in_all_student_household_age_30_over_perc', 
 'living_alone_age_4_under_count',
 'living_alone_age_5_15_count',
 'living_alone_age_16_17_count',
 'living_alone_age_18_20_count',
 'living_alone_age_21_24_count',
 'living_alone_age_25_29_count',
 'living_alone_age_30_over_count',
 'living_alone_age_4_under_perc',
 'living_alone_age_5_15_perc',
 'living_alone_age_16_17_perc',
 'living_alone_age_18_20_perc',
 'living_alone_age_21_24_perc',
 'living_alone_age_25_29_perc',
 'living_alone_age_30_over_perc', 
 'living_in_other_household_age_4_under_count',
 'living_in_other_household_age_5_15_count',
 'living_in_other_household_age_16_17_count',
 'living_in_other_household_age_18_20_count',
 'living_in_other_household_age_21_24_count',
 'living_in_other_household_age_25_29_count',
 'living_in_other_household_age_30_over_count',
 'living_in_other_household_age_4_under_perc',
 'living_in_other_household_age_5_15_perc',
 'living_in_other_household_age_16_17_perc',
 'living_in_other_household_age_18_20_perc',
 'living_in_other_household_age_21_24_perc',
 'living_in_other_household_age_25_29_perc',
 'living_in_other_household_age_30_over_perc',
 ]

census2021_student_lsoa2021_gdb_df = census2021_student_lsoa2021_gdb_df[final_column_order]

census2021_student_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household

# 8. Create dominant field

In [24]:
def determine_dominant_age_group(row):
    age_columns = [
         'age_4_under_perc',
         'age_5_15_perc',
         'age_16_17_perc',
         'age_18_20_perc',
         'age_21_24_perc',
         'age_25_29_perc',
         'age_30_over_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_student_lsoa2021_gdb_df['dominant_student_age_group'] = census2021_student_lsoa2021_gdb_df.apply(determine_dominant_age_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_student_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household

In [25]:
def determine_dominant_age_group(row):
    age_columns = [
         'living_with_parents_perc',
         'living_in_communal_establishment_uni_perc',
         'living_in_communal_establishment_other_perc',
         'living_in_all_student_household_perc',
         'living_alone_perc',
         'living_in_other_household_perc'
    ]
    
    max_value = max(row[col] for col in age_columns)
    threshold = max_value - threshold_value
    
    dominant_groups = [col.replace('_perc', '') for col in age_columns if row[col] >= threshold]
    
    return ', '.join(dominant_groups)

census2021_student_lsoa2021_gdb_df['dominant_student_household_group'] = census2021_student_lsoa2021_gdb_df.apply(determine_dominant_age_group, axis=1)

# Display the first few rows of the updated dataframe
census2021_student_lsoa2021_gdb_df.head()

,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household

# Special Case: Find and remove Duplicates

In [26]:
def remove_duplicates(df, unique_column):
    """
    Removes duplicate rows based on a specified unique column while keeping the first occurrence.
    
    Parameters:
        df (GeoDataFrame or DataFrame): The input data to clean.
        unique_column (str): The column to check for duplicates.
    
    Returns:
        GeoDataFrame or DataFrame: A cleaned dataframe with duplicates removed.
    """
    # Check for duplicates
    duplicate_count = df.duplicated(subset=[unique_column], keep='first').sum()
    
    if duplicate_count > 0:
        print(f"Found {duplicate_count} duplicate(s) in column '{unique_column}'. Removing duplicates...")

        # Remove duplicates, keeping the first occurrence
        df = df.drop_duplicates(subset=[unique_column], keep='first')

        print(f"Duplicates removed. Now {unique_column} has only unique values.")

    else:
        print(f"No duplicates found in column '{unique_column}'. No changes made.")

    return df

# Specify the column where duplicates should be checked
unique_column = "lsoa21cd"

# Remove duplicates before uploading to PostGIS
census2021_student_lsoa2021_gdb_df = remove_duplicates(census2021_student_lsoa2021_gdb_df, unique_column)

census2021_student_lsoa2021_gdb_df.head()

No duplicates found in column 'lsoa21cd'. No changes made.


,FID,lsoa21cd,lsoa21nm,geometry,msoa21cd,msoa21nm,lad22cd,lad22nm,wd22cd,wd22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_students,age_4_under_count,age_5_15_count,age_16_17_count,age_18_20_count,age_21_24_count,age_25_29_count,age_30_over_count,age_4_under_perc,age_5_15_perc,age_16_17_perc,age_18_20_perc,age_21_24_perc,age_25_29_perc,age_30_over_perc,living_with_parents_count,living_in_communal_establishment_uni_count,living_in_communal_establishment_other_count,living_in_all_student_household_count,living_alone_count,living_in_other_household_count,living_with_parents_perc,living_in_communal_establishment_uni_perc,living_in_communal_establishment_other_perc,living_in_all_student_household_perc,living_alone_perc,living_in_other_household_perc,living_with_parents_age_4_under_count,living_with_parents_age_5_15_count,living_with_parents_age_16_17_count,living_with_parents_age_18_20_count,living_with_parents_age_21_24_count,living_with_parents_age_25_29_count,living_with_parents_age_30_over_count,living_with_parents_age_4_under_perc,living_with_parents_age_5_15_perc,living_with_parents_age_16_17_perc,living_with_parents_age_18_20_perc,living_with_parents_age_21_24_perc,living_with_parents_age_25_29_perc,living_with_parents_age_30_over_perc,living_in_communal_establishment_uni_age_4_under_count,living_in_communal_establishment_uni_age_5_15_count,living_in_communal_establishment_uni_age_16_17_count,living_in_communal_establishment_uni_age_18_20_count,living_in_communal_establishment_uni_age_21_24_count,living_in_communal_establishment_uni_age_25_29_count,living_in_communal_establishment_uni_age_30_over_count,living_in_communal_establishment_uni_age_4_under_perc,living_in_communal_establishment_uni_age_5_15_perc,living_in_communal_establishment_uni_age_16_17_perc,living_in_communal_establishment_uni_age_18_20_perc,living_in_communal_establishment_uni_age_21_24_perc,living_in_communal_establishment_uni_age_25_29_perc,living_in_communal_establishment_uni_age_30_over_perc,living_in_communal_establishment_other_age_4_under_count,living_in_communal_establishment_other_age_5_15_count,living_in_communal_establishment_other_age_16_17_count,living_in_communal_establishment_other_age_18_20_count,living_in_communal_establishment_other_age_21_24_count,living_in_communal_establishment_other_age_25_29_count,living_in_communal_establishment_other_age_30_over_count,living_in_communal_establishment_other_age_4_under_perc,living_in_communal_establishment_other_age_5_15_perc,living_in_communal_establishment_other_age_16_17_perc,living_in_communal_establishment_other_age_18_20_perc,living_in_communal_establishment_other_age_21_24_perc,living_in_communal_establishment_other_age_25_29_perc,living_in_communal_establishment_other_age_30_over_perc,living_in_all_student_household_age_4_under_count,living_in_all_student_household_age_5_15_count,living_in_all_student_household_age_16_17_count,living_in_all_student_household_age_18_20_count,living_in_all_student_household_age_21_24_count,living_in_all_student_household_age_25_29_count,living_in_all_student_household_age_30_over_count,living_in_all_student_household_age_4_under_perc,living_in_all_student_household_age_5_15_perc,living_in_all_student_household_age_16_17_perc,living_in_all_student_household_age_18_20_perc,living_in_all_student_household_age_21_24_perc,living_in_all_student_household_age_25_29_perc,living_in_all_student_household_age_30_over_perc,living_alone_age_4_under_count,living_alone_age_5_15_count,living_alone_age_16_17_count,living_alone_age_18_20_count,living_alone_age_21_24_count,living_alone_age_25_29_count,living_alone_age_30_over_count,living_alone_age_4_under_perc,living_alone_age_5_15_perc,living_alone_age_16_17_perc,living_alone_age_18_20_perc,living_alone_age_21_24_perc,living_alone_age_25_29_perc,living_alone_age_30_over_perc,living_in_other_household_age_4_under_count,living_in_other_household_age_5_15_count,living_in_other_household

# 9. Upload to geodatabase

In [27]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "census2021_lsoa2021_students"  # Desired table name
primary_key_column = "lsoa21cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [28]:
# Ensure the GeoDataFrame has a valid CRS before writing
if census2021_student_lsoa2021_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    census2021_student_lsoa2021_gdb_df.set_crs(epsg=27700, inplace=True)

In [29]:
# Publish the GeoDataFrame to PostGIS
census2021_student_lsoa2021_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.census2021_lsoa2021_students
Primary key set on column: lsoa21cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.census2021_lsoa2021_students
